# Aula 06 — Realimentação de Relevância: Rocchio, Expansão de Consulta e Pseudo-Realimentação (Teoria)
**Projeto Integrador III — Motor de Busca**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel)*

In [1]:
# 1. Setup oficial do motor06 e inspecao do estado herdado
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")
docs <- docs_aula()
cfg  <- cfg_aula()
ix   <- montar(docs, cfg)
estado()

[1] "MOTOR_VERSAO: motor06 v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 6 | elementos: limpar, acentos, stopwords, k1, b, limiar |
| `docs` | character, nomeado | 8 | d1 recuperacao de informacao ordena..., d2 o modelo de espaco vetorial repre..., ... |
| `ix` | list | 11 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn, postings, dl, avgdl, idf_bm25 |


## Módulo 1 — A primeira consulta raramente é perfeita
Execução do ranking base do cosseno, medição com o gabarito (AP = 0,5) e identificação dos documentos relevantes e irrelevantes no top-4.

In [2]:
# Módulo 1: Ranking base do cosseno e avaliacao inicial
base <- ranking_cosseno("modelo de recuperacao", ix, cfg)
cat("Ranking base do cosseno:\n")
print(round(base, 3))

grau <- gabarito_aula()
cat("\nMetricas iniciais do ranking base (limiar 2):\n")
print(round(metricas(base, grau, 2), 3))

Ranking base do cosseno:
   d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000 

Metricas iniciais do ranking base (limiar 2):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.333     0.500     0.500     0.651     0.837 


## Módulo 2 — A consulta é um vetor; dá para movê-la
Cálculo de centroides vetoriais no espaço TF-IDF de 45 dimensões via `rowMeans()` e `drop = FALSE`.

In [3]:
# Módulo 2: Vetor de consulta q0 e centroide dos relevantes
q0 <- vetor_consulta(tokenizar("modelo de recuperacao"), ix$vocab, ix$idf_tfidf)
cat("Coordenadas nao-nulas da consulta original q0:\n")
print(round(q0[q0 > 0], 3))

cr <- rowMeans(ix$w[, c("d2", "d3")])
cat("\nPesos no centroide cr (de, modelo, espaco):\n")
print(round(cr[c("de", "modelo", "espaco")], 3))

# Checkpoint 2: centroide de d1, d3, d4 em recuperacao
cat("\nCheckpoint 2 - Centroide de d1, d3, d4 em 'recuperacao':\n")
print(round(rowMeans(ix$w[, c("d1", "d3", "d4")])[["recuperacao"]], 3))

Coordenadas nao-nulas da consulta original q0:
         de      modelo recuperacao 
      0.470       1.386       1.386 

Pesos no centroide cr (de, modelo, espaco):
    de modelo espaco 
 0.705  1.386  1.040 

Checkpoint 2 - Centroide de d1, d3, d4 em 'recuperacao':
[1] 0.924


## Módulo 3 — A fórmula de Rocchio e a expansão de consulta
Aplicação da fórmula canônica de Rocchio: atração em direção aos relevantes ($\\beta = 0{,}75$), repulsão dos irrelevantes ($\\gamma = 0{,}15$) e expansão de 3 para 21 termos.

In [4]:
# Módulo 3: Formula canônica de Rocchio
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0
  a * q + b * cr - g * cnr
}

qm <- rocchio(q0, Dr = c("d2", "d3"), Dnr = "d4", ix = ix)
cat("Total de termos com peso nao-nulo na consulta reformulada qm:", sum(qm != 0), "\n")

cat("\nTop 8 termos de maior peso positivo em qm (expansao de consulta):\n")
print(round(head(sort(qm, decreasing = TRUE), 8), 3))

cat("\nTermos com peso negativo (repulsao de d4):\n")
print(round(qm[qm < 0], 3))

Total de termos com peso nao-nulo na consulta reformulada qm: 21 

Top 8 termos de maior peso positivo em qm (expansao de consulta):
        modelo    recuperacao             de           bm25           como 
         2.426          1.178          0.999          0.780          0.780 
        espaco probabilistico   ranqueamento 
         0.780          0.780          0.780 

Termos com peso negativo (repulsao de d4):
          a aprendizado estatistico  fundamenta     moderna 
     -0.104      -0.312      -0.312      -0.312      -0.312 


## Módulo 4 — O novo ranking
Busca vetorial reformulada via `ranking_vetor()`: cossenos negativos, reposicionamento de d2 e d4, e salto de AP de 0,5 para 1,0.

In [5]:
# Módulo 4: Ranqueamento vetorial e salto de metricas
ranking_vetor <- function(q, ix) {
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), decreasing = TRUE)
}

novo <- ranking_vetor(qm, ix)
cat("Novo ranking com a consulta reformulada qm:\n")
print(round(novo, 3))

cat("\nComparacao das metricas (Base vs Rocchio):\n")
tab_comp <- round(rbind(base = metricas(base, grau, 2), rocchio = metricas(novo, grau, 2)), 3)
print(tab_comp)

# Teste com pmax (sem pesos negativos)
novo_pmax <- ranking_vetor(pmax(qm, 0), ix)
cat("\nMetricas com pmax(qm, 0) (Checkpoint 4):\n")
print(round(metricas(novo_pmax, grau, 2), 3))

Novo ranking com a consulta reformulada qm:
    d3     d2     d1     d6     d8     d5     d7     d4 
 0.650  0.643  0.140  0.055  0.045  0.041 -0.007 -0.060 

Comparacao das metricas (Base vs Rocchio):
        P_at_k  AP  RR nDCG_bin nDCG_grad
base     0.333 0.5 0.5    0.651     0.837
rocchio  0.667 1.0 1.0    1.000     1.000

Metricas com pmax(qm, 0) (Checkpoint 4):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.667     1.000     1.000     1.000     0.990 


## Módulo 5 — Avaliar com honestidade: a coleção residual
Exclusão dos documentos já conhecidos ($D_r \cup D_{nr}$) para medir se a consulta reformulada é capaz de recuperar novos documentos relevantes.

In [6]:
# Módulo 5: Avaliacao honesta na colecao residual
fora  <- c("d2", "d3", "d4")
g_res <- grau[!names(grau) %in% fora]
b_res <- base[!names(base) %in% fora]
n_res <- novo[!names(novo) %in% fora]

cat("Metricas na colecao residual com limiar 2 (R = 0 -> AP NaN/NA):\n")
print(round(metricas(b_res, g_res, 2), 3))

cat("\nMetricas na colecao residual com limiar 1 (Empate nos documentos nao vistos):\n")
tab_res1 <- round(rbind(base = metricas(b_res, g_res, 1), rocchio = metricas(n_res, g_res, 1)), 3)
print(tab_res1)

Metricas na colecao residual com limiar 2 (R = 0 -> AP NaN/NA):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
        0        NA         0        NA         1 

Metricas na colecao residual com limiar 1 (Empate nos documentos nao vistos):
        P_at_k AP RR nDCG_bin nDCG_grad
base     0.667  1  1        1         1
rocchio  0.667  1  1        1         1


## Módulo 6 — Pseudo-realimentação (PRF) e o query drift
Realimentação cega automática assumindo os top-$k$ como relevantes e demonstração do fenômeno de *query drift* (deriva temática).

In [7]:
# Módulo 6: Pseudo-relevance feedback (PRF)
top2 <- names(base)[1:2]
q_prf2 <- rocchio(q0, top2, ix = ix)
r_prf2 <- ranking_vetor(q_prf2, ix)

cat("PRF com k = 2 (Dr = d1, d3):\n")
print(round(r_prf2, 3))
cat("\nMetricas com k = 2 (AP sobe de 0.5 para 0.833 sem usuario!):\n")
print(round(metricas(r_prf2, grau, 2), 3))

# Drift com k = 3
top3 <- names(base)[1:3]
q_prf3 <- rocchio(q0, top3, ix = ix)
r_prf3 <- ranking_vetor(q_prf3, ix)
cat("\nMetricas com k = 3 (inclui d4 irrelevante -> query drift! AP cai para 0.750):\n")
print(round(metricas(r_prf3, grau, 2), 3))

# Drift severo com k = 1 (Checkpoint 6)
top1 <- names(base)[1]
q_prf1 <- rocchio(q0, top1, ix = ix)
r_prf1 <- ranking_vetor(q_prf1, ix)
cat("\nCheckpoint 6 - Metricas com k = 1 (apenas d1 -> AP cai para 0.417):\n")
print(round(metricas(r_prf1, grau, 2), 3))

PRF com k = 2 (Dr = d1, d3):
   d3    d1    d2    d4    d6    d8    d7    d5 
0.648 0.575 0.175 0.161 0.062 0.047 0.037 0.010 

Metricas com k = 2 (AP sobe de 0.5 para 0.833 sem usuario!):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.667     0.833     1.000     0.920     0.958 

Metricas com k = 3 (inclui d4 irrelevante -> query drift! AP cai para 0.750):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.333     0.750     1.000     0.877     0.925 

Checkpoint 6 - Metricas com k = 1 (apenas d1 -> AP cai para 0.417):
   P_at_k        AP        RR  nDCG_bin nDCG_grad 
    0.333     0.417     0.500     0.686     0.841 


## Fechamento da Sessão e Snapshot de Estado
Anexação da memória do R no consolidado.

In [8]:
# Fechamento e persistencia de estado
if (file.exists("aula06_consolidado.md")) {
  anexar_estado("aula06_consolidado.md")
  cat("Estado anexado a aula06_consolidado.md com sucesso!\n")
}
cat("Sessao teorica da Aula 06 finalizada!\n")

Estado anexado a aula06_consolidado.md com sucesso!
Sessao teorica da Aula 06 finalizada!
